# Week 10 Discussion: Ray + Spark Integration

**Course:** DSC 232R - Big Data Analysis Using Spark

Welcome to Week 10! Throughout this course, you've built expertise with Spark for data processing, SQL, and initial ML tasks. Today, we bridge the gap between Data Engineering and Machine Learning by integrating Spark with Ray.

### Agenda
1. **Environment Setup**
2. **The Landscape:** Spark vs. Ray Comparison
3. **Approach 1: RayDP (Spark on Ray)**
4. **Approach 2: Data Handoff Patterns**
5. **Wrap-up & Exercises**

In [ ]:
# 0. Environment Setup
# Installing required packages. We are using modern PySpark to support Python 3.12+
!pip install -q raydp pyspark pyarrow xgboost ray
print("Required packages installed successfully!")

# IMPORTANT: If you just ran this for the first time, restart your kernel/runtime now

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.4/317.4 MB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.7/406.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 MB 9.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.7 which is incompatible.
Required packages installed successfully!


In [ ]:
import ray
import numpy as np
import pandas as pd
import time
import os
import tempfile
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Check RayDP availability for the live demo
try:
    import raydp
    RAYDP_AVAILABLE = True
    print(f"RayDP available: version {raydp.__version__}")
except ImportError:
    RAYDP_AVAILABLE = False
    print("RayDP not installed. Using standard fallback modes for demonstration.")

TEMP_DIR = tempfile.mkdtemp(prefix="dsc232r_week10_")
print(f"Temp directory created at: {TEMP_DIR}")

RayDP available: version 1.6.4
Temp directory created at: /tmp/dsc232r_week10_z48pbqtm


## 1. Spark vs. Ray: Choosing the Right Tool

It's not about choosing Spark **OR** Ray; it's about using Spark **AND** Ray where they respectively excel.

| Feature | Apache Spark | Ray |
| :--- | :--- | :--- |
| **Primary Strength** | Complex SQL, ETL, Heavy Joins | ML Training, Stateful Actors, Custom Python |
| **Data Model** | DataFrame / RDD | Tasks / Actors / Datasets |
| **Python UDFs** | Slower (due to JVM serialization) | Extremely fast (native Python/Arrow) |

> **The Golden Rule** Use Spark for structured data manipulation and ETL. Hand off to Ray for machine learning, hyperparameter tuning, and custom Python algorithms.

In [ ]:
# Initialize Basic Ray and Spark for Comparison
if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus=4, logging_level="WARNING")

spark = SparkSession.builder \
    .appName("SparkRayComparison") \
    .master("local[2]") \
    .getOrCreate()

# 1. Generate Dummy Data
n_rows = 100000
data = pd.DataFrame({
    "id": range(n_rows),
    "value": np.random.randn(n_rows),
    "category": np.random.choice(["A", "B", "C"], n_rows)
})

# 2. Spark Transformation Example (Declarative)
spark_df = spark.createDataFrame(data)
spark_res = spark_df.groupBy("category").agg(F.avg("value").alias("avg_val"))
print("--- Spark Aggregation ---")
spark_res.show(3)

# 3. Ray Transformation Example (Functional / Pythonic)
ray_ds = ray.data.from_pandas(data)
ray_res = ray_ds.groupby("category").mean("value")
print("--- Ray Aggregation ---")
ray_res.show(3)

# Cleanup for next section
spark.stop()
ray.shutdown()

/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


--- Spark Aggregation ---
+--------+--------------------+
|category|             avg_val|
+--------+--------------------+
|       B| 5.17079435919224E-4|
|       C|-9.10061525503819...|
|       A|-8.34640268367485...|
+--------+--------------------+



2026-03-10 22:01:33,065	INFO dataset.py:3670 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-03-10 22:01:33,084	INFO logging.py:392 -- Registered dataset logger for dataset dataset_2_0
2026-03-10 22:01:33,108	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=1, aggregators=1, dataset (estimate)=0.0GiB): shuffle=6.3MiB, output=6.3MiB, total=12.6MiB, 
2026-03-10 22:01:33,115	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_2_0. Full logs are in /tmp/ray/session_2026-03-10_22-00-37_313171_196/logs/ray-data
2026-03-10 22:01:33,117	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_2_0: InputDataBuffer[Input] -> HashAggregateOperator[HashAggregate(key_columns=('category',), num_partitions=1)] -> LimitOperator[limit=3]
2026-03-10 22:01:33,142	WARNING resource_manager.py:141 -- ⚠️  Ray's object store is configured to use only 42.9% of availabl

--- Ray Aggregation ---


2026-03-10 22:01:38,206	WARNING default_autoscaling_coordinator.py:134 -- Failed to send resource request for data-dataset_2_0. If this only happens transiently during network partition or CPU being overloaded, it's safe to ignore this error. If this error persists, file a GitHub issue.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/ray/data/_internal/cluster_autoscaler/default_autoscaling_coordinator.py", line 98, in wrapper
    result = func(self, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ray/data/_internal/cluster_autoscaler/default_autoscaling_coordinator.py", line 181, in request_resources
    ray.get(
  File "/usr/local/lib/python3.12/dist-packages/ray/_private/auto_init_hook.py", line 22, in auto_init_wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ray/_private/client_mode_hook.py", line 104, in wrapper
    retu

{'category': 'A', 'mean(value)': -8.346402683673578e-05}
{'category': 'B', 'mean(value)': 0.0005170794359192278}
{'category': 'C', 'mean(value)': -9.100615255036886e-05}


## 2. Approach 1: RayDP (Spark on Ray)

**RayDP** allows you to run Spark executors natively as Ray actors. This means you have **unified resource management** and a **shared object store**.

### Why is this powerful?
Instead of serializing data and writing it to disk to pass from Spark to Ray, RayDP allows for **zero-copy** in-memory data transfer via Apache Arrow.



In [ ]:
# Initialize Ray
ray.init(num_cpus=4, logging_level="WARNING")

if RAYDP_AVAILABLE:
    try:
        # Attempt to initialize Spark on Ray
        spark = raydp.init_spark(
            app_name="RayDP_Demo",
            num_executors=2,
            executor_cores=1,
            executor_memory="1GB"
        )
        print("Spark is now running natively on Ray!")
    except Exception as e:
        # Safety net for the live demo
        print("RayDP failed to initialize the Spark context (often due to Python/Spark version mismatch).")
        print("Falling back to standard local Spark.\n")
        RAYDP_AVAILABLE = False
        spark = SparkSession.builder.appName("Fallback").master("local[2]").getOrCreate()
        print("Running local Spark (Fallback mode).")
else:
    spark = SparkSession.builder.appName("Fallback").master("local[2]").getOrCreate()
    print("Running local Spark (Fallback mode).")

# --- THE UNIFIED PIPELINE ---

# Step A: Spark ETL
raw_df = spark.createDataFrame(data)
cleaned_df = raw_df.filter(F.col("value") > 0)

# Step B: ZERO-COPY HANDOFF to Ray Dataset
if RAYDP_AVAILABLE:
    print("\nExecuting Zero-Copy Transfer using RayDP...")
    ray_dataset = ray.data.from_spark(cleaned_df)
else:
    print("\nExecuting standard transfer via Pandas...")
    ray_dataset = ray.data.from_pandas(cleaned_df.toPandas())

print(f"Ray Dataset Count: {ray_dataset.count()}")

# Step C: Ray Machine Learning (Simulated Feature Engineering)
def engineer_features(batch: pd.DataFrame) -> pd.DataFrame:
    batch["value_squared"] = batch["value"] ** 2
    return batch

ml_dataset = ray_dataset.map_batches(engineer_features, batch_format="pandas")
print("\nTransformed ML Dataset Sample:")
ml_dataset.show(3)

(RayDPSparkMaster pid=2022) [2026-03-10 22:03:19,882 I 2122 2158] gcs_client.cc:96: GcsClient has no Cluster ID set, and won't fetch from GCS.
(RayDPSparkMaster pid=2022) [2026-03-10 22:03:20,140 I 2122 2158] gcs_client.cc:96: GcsClient has no Cluster ID set, and won't fetch from GCS.
(raylet) [2026-03-10 22:03:27,170 I 2189 2206] gcs_client.cc:96: GcsClient has no Cluster ID set, and won't fetch from GCS.


RayDP failed to initialize the Spark context (often due to Python/Spark version mismatch).
Falling back to standard local Spark.

Running local Spark (Fallback mode).

Executing standard transfer via Pandas...


2026-03-10 22:03:44,292	INFO logging.py:392 -- Registered dataset logger for dataset dataset_2_0
2026-03-10 22:03:44,299	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_2_0. Full logs are in /tmp/ray/session_2026-03-10_22-02-51_802511_196/logs/ray-data
2026-03-10 22:03:44,301	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_2_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(engineer_features)] -> LimitOperator[limit=3]


Ray Dataset Count: 49895

Transformed ML Dataset Sample:


2026-03-10 22:03:47,968	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_2_0 =======
2026-03-10 22:03:47,970	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-03-10 22:03:47,973	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-03-10 22:03:47,976	INFO logging_progress.py:181 -- 
2026-03-10 22:03:47,980	INFO logging_progress.py:231 -- MapBatches(engineer_features): 0/1
2026-03-10 22:03:47,984	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 0 (0.0B); Resources: 1.0 CPU, 384.0MiB object store
2026-03-10 22:03:47,987	INFO logging_progress.py:231 -- limit=3: 0/1
2026-03-10 22:03:47,991	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-03-10 22:03:47,995	INFO logging_progress.py:192 -- ============================================
2026-03-10 22:03:48,343	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_2_0 execution finis

{'id': 0, 'value': 1.0471595884210922, 'category': 'C', 'value_squared': 1.0965432036222313}
{'id': 1, 'value': 1.121279168818673, 'category': 'C', 'value_squared': 1.257266974426694}
{'id': 3, 'value': 0.681522739970709, 'category': 'A', 'value_squared': 0.46447324509718263}


## 3. Approach 2: Data Handoff Patterns

If you are not using RayDP (e.g., you have a standalone Spark cluster and a standalone Ray cluster), you need to physically move data between them.

### The Three Strategies
1. **In-Memory (Pandas):** Fast, but highly constrained. Entire dataset must fit in the Spark Driver memory. *Only for <10GB data.*
2. **Arrow Format:** Efficient, columnar in-memory/disk transfer.
3. **Parquet Files (Recommended):** The industry standard. Spark writes Parquet to distributed storage (S3/HDFS), Ray reads it. Preserves schema, allows compression, and scales infinitely.



In [ ]:
def benchmark_handoff():
    # Create a medium dataset for the benchmark
    test_df = spark.createDataFrame(pd.DataFrame({
        "feature1": np.random.randn(500000),
        "feature2": np.random.randn(500000)
    }))

    print("Benchmarking Handoff Patterns...\n")

    # Pattern 1: Pandas (In-Memory)
    start = time.time()
    pandas_df = test_df.toPandas() # DANGER: Pulls all data to driver
    ds_pandas = ray.data.from_pandas(pandas_df)
    _ = ds_pandas.count()
    print(f"Pandas In-Memory Transfer: {time.time() - start:.3f} seconds")

    # Pattern 2: Parquet (Disk)
    start = time.time()
    parquet_path = os.path.join(TEMP_DIR, "benchmark.parquet")
    test_df.write.mode("overwrite").parquet(parquet_path)
    ds_parquet = ray.data.read_parquet(parquet_path)
    _ = ds_parquet.count()
    print(f"Parquet File Exchange:     {time.time() - start:.3f} seconds")

benchmark_handoff()

# Cleanup the session
if RAYDP_AVAILABLE:
    raydp.stop_spark()
else:
    spark.stop()
ray.shutdown()

Benchmarking Handoff Patterns...

Pandas In-Memory Transfer: 6.898 seconds


Parquet dataset sampling:   0%|          | 0.00/2.00 [00:00<?, ? file/s]

2026-03-10 22:04:10,854	INFO parquet_datasource.py:1079 -- Estimated parquet encoding ratio is 1.062.
2026-03-10 22:04:10,862	INFO parquet_datasource.py:1139 -- Estimated parquet reader batch size at 7895161 rows
2026-03-10 22:04:10,905	INFO logging.py:392 -- Registered dataset logger for dataset dataset_5_0
2026-03-10 22:04:10,943	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_5_0. Full logs are in /tmp/ray/session_2026-03-10_22-02-51_802511_196/logs/ray-data
2026-03-10 22:04:10,945	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_5_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> AggregateNumRows[AggregateNumRows]
2026-03-10 22:04:11,250	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_5_0 =======
2026-03-10 22:04:11,256	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-03-10 22:04:11,270	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-03-10 22:04:11

Parquet File Exchange:     8.090 seconds


## 4. Summary & Discussion Exercises

### Decision Matrix
| Strategy | Data Size | Latency | Memory Use | Setup Complexity |
|----------|-----------|---------|------------|-------|
| **Parquet** | Any | Higher | Low | Simple |
| **In-Memory (Pandas)** | < 10 GB | Lowest | High | Simple |
| **RayDP** | Any | Lowest | Medium | Complex |

---

### Class Exercises (Let's discuss!)

**Scenario 1:**
You have a 500GB dataset of web logs. You need to join it with user profiles and calculate aggregate session times. Should you use Spark or Ray?
*Answer:* Spark. It natively handles complex joins and aggregations on large structured datasets using disk-spill and shuffle optimizations.

**Scenario 2:**
Your Spark DataFrame has 50 million rows of engineered features. You need to train a distributed XGBoost model on it. Which handoff strategy do you choose?
*Answer:* Parquet. 50 million rows is too large for a safe `toPandas()` driver collection. Writing to partitioned Parquet allows Ray workers to read the data in parallel.

**Scenario 3:**
You want to pass a 500 MB lookup table from Spark to Ray to do some interactive data science. Which handoff strategy do you choose?
*Answer:* In-memory via Pandas or Arrow. It's small enough to fit safely in memory and minimizes interactive latency.
